In [29]:
from utils.DataPreprocessing import dataloader
from loguru import logger
import os

lag = 20

path = os.path.dirname(os.path.dirname(os.getcwd()))
print(path)
data = dataloader(path=path, lag=lag)

2024-04-13 05:10:00.944 | INFO     | utils.DataPreprocessing:load_csv:129 - shape: (11_103, 3)
┌───────────┬─────────────┬────────────┐
│ AVG_TEMP  ┆ AVG_TEMP_DC ┆ date       │
│ ---       ┆ ---         ┆ ---        │
│ f32       ┆ str         ┆ date       │
╞═══════════╪═════════════╪════════════╡
│ 29.299999 ┆ C           ┆ 1992-07-01 │
│ 29.200001 ┆ C           ┆ 1992-07-02 │
│ 29.6      ┆ C           ┆ 1992-07-03 │
│ 29.299999 ┆ C           ┆ 1992-07-04 │
│ 29.299999 ┆ C           ┆ 1992-07-05 │
│ …         ┆ …           ┆ …          │
│ 22.200001 ┆ C           ┆ 2022-11-26 │
│ 22.4      ┆ C           ┆ 2022-11-27 │
│ 25.4      ┆ C           ┆ 2022-11-28 │
│ 25.1      ┆ C           ┆ 2022-11-29 │
│ 22.1      ┆ C           ┆ 2022-11-30 │
└───────────┴─────────────┴────────────┘
2024-04-13 05:10:00.945 | INFO     | utils.DataPreprocessing:load_csv:131 - shape: (1, 3)
┌──────────┬─────────────┬──────┐
│ AVG_TEMP ┆ AVG_TEMP_DC ┆ date │
│ ---      ┆ ---         ┆ ---  │
│ u32      ┆ u32

/home/argonaut/programming/HK-Temperature-Forecasting


In [30]:
print(data.X_train[0].columns)

['AVG_TEMP', 'GSR', 'SUN', 'RH', 'UV', 'RF', 'WSPD', 'weather', 'dayofweek', 'Month', 'year', 'dayofmonth', 'weekofyear', 'dayofyear', 'quarter']


In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import tensorflow as tf

n_past = lag
n_future = 1
n_features = 15
output_features = 8

In [32]:
print(data.X_train[0])

shape: (20, 15)
┌───────────┬───────────┬──────┬──────┬───┬────────────┬────────────┬───────────┬─────────┐
│ AVG_TEMP  ┆ GSR       ┆ SUN  ┆ RH   ┆ … ┆ dayofmonth ┆ weekofyear ┆ dayofyear ┆ quarter │
│ ---       ┆ ---       ┆ ---  ┆ ---  ┆   ┆ ---        ┆ ---        ┆ ---       ┆ ---     │
│ f32       ┆ f32       ┆ f32  ┆ f32  ┆   ┆ f32        ┆ f32        ┆ f32       ┆ f32     │
╞═══════════╪═══════════╪══════╪══════╪═══╪════════════╪════════════╪═══════════╪═════════╡
│ 29.200001 ┆ 25.1      ┆ 11.6 ┆ 79.0 ┆ … ┆ 1.0        ┆ 30.0       ┆ 213.0     ┆ 3.0     │
│ 29.1      ┆ 15.13     ┆ 7.0  ┆ 82.0 ┆ … ┆ 2.0        ┆ 31.0       ┆ 214.0     ┆ 3.0     │
│ 29.0      ┆ 11.85     ┆ 2.4  ┆ 83.0 ┆ … ┆ 3.0        ┆ 31.0       ┆ 215.0     ┆ 3.0     │
│ 27.5      ┆ 10.67     ┆ 2.3  ┆ 87.0 ┆ … ┆ 4.0        ┆ 31.0       ┆ 216.0     ┆ 3.0     │
│ 27.9      ┆ 15.18     ┆ 6.0  ┆ 82.0 ┆ … ┆ 5.0        ┆ 31.0       ┆ 217.0     ┆ 3.0     │
│ …         ┆ …         ┆ …    ┆ …    ┆ … ┆ …          ┆ …      

In [33]:
import polars as pl
X = data.X_train
X_train = np.array([])
for df in X:
    df = pl.DataFrame(df)
    df = df.to_numpy()
    df = df.reshape((1, df.shape[0], df.shape[1]))
    # print(df.shape)
    if X_train.shape[0] == 0:
        X_train = df
    else:
        X_train = np.vstack((X_train, df))

print(X_train.shape)


(6798, 20, 15)


In [34]:
y = data.y_train   
y_train = np.array([])
for df in y:
    df = pl.DataFrame(df)
    df = df.to_numpy()
    # df = df.reshape((1, df.shape[0], df.shape[1]))
    # print(df.shape)
    if y_train.shape[0] == 0:
        y_train = df
    else:
        y_train = np.vstack((y_train, df))

print(y_train.shape)

(6798, 8)


In [35]:
X = data.X_test
X_test = np.array([])
for df in X:
    df = pl.DataFrame(df)
    df = df.to_numpy()
    df = df.reshape((1, df.shape[0], df.shape[1]))
    # print(df.shape)
    if X_test.shape[0] == 0:
        X_test = df
    else:
        X_test = np.vstack((X_test, df))
        
print(X_test.shape)

(1685, 20, 15)


In [36]:
y = data.y_test
y_test = np.array([])
for df in y:
    df = pl.DataFrame(df)
    df = df.to_numpy()
    # df = df.reshape((1, df.shape[0], df.shape[1]))
    # print(df.shape)
    if y_test.shape[0] == 0:
        y_test = df
    else:
        y_test = np.vstack((y_test, df))
        
print(y_test.shape)

(1685, 8)


In [37]:
# E1D1
# n_features ==> no of features at each timestep in the data.
#
import keras

hidden_size = 50
encoder_inputs = keras.layers.Input(shape=(n_past, n_features))
encoder_l1 = keras.layers.LSTM(hidden_size, return_state=True)
encoder_outputs1 = encoder_l1(encoder_inputs)

encoder_states1 = encoder_outputs1[1:]

#
decoder_inputs = keras.layers.RepeatVector(n_future)(encoder_outputs1[0])

#
decoder_l1 = keras.layers.LSTM(hidden_size, return_sequences=True)(
    decoder_inputs, initial_state=encoder_states1
)
decoder_outputs1 = keras.layers.TimeDistributed(keras.layers.Dense(output_features))(
    decoder_l1
)

#
model_e1d1 = keras.models.Model(encoder_inputs, decoder_outputs1)

#
model_e1d1.summary()

Model: "functional_36"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 20, 15)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_16 (LSTM)      │ [(None, 50),      │     13,200 │ input_layer_6[0]… │
│                     │ (None, 50),       │            │                   │
│                     │ (None, 50)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_1     │ (None, 1, 50)     │          0 │ lstm_16[0][0]     │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_17 (LSTM)      │ (None, 1, 50)     │     20,200 │ repeat_vector_1[… │
│                     │                   │            │ lstm_16[0][1],    │
│                     │                   │            │ lstm_16[0][2]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 1, 8)      │        408 │ lstm_17[0][0]     │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 33,808 (132.06 KB)

 Trainable params: 33,808 (132.06 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
# reduce_lr = keras.callbacks.LearningRateScheduler(lambda x: 1e-3 * 0.90**x)
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, mode='auto')
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, mode='auto')
model_e1d1.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=keras.losses.Huber())

In [39]:
# Building the RNN
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout

regressor = Sequential()

# Addinf the first LSTM layer and some Dropout regularisation
regressor.add(keras.layers.Input(shape=(n_past, n_features), batch_size=1))

regressor.add(LSTM(units=hidden_size, return_sequences=True))
regressor.add(Dropout(0.5))

# regressor.add(LSTM(units=hidden_size, return_sequences=True))
# regressor.add(Dropout(0.5))

# regressor.add(LSTM(units=hidden_size, return_sequences=True))
# regressor.add(Dropout(0.5))

regressor.add(LSTM(units=hidden_size))
regressor.add(Dropout(0.5))

# Output layer
regressor.add(Dense(units=output_features))
regressor.summary()

regressor.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="mean_squared_error",
)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_18 (LSTM)                  │ (1, 20, 50)            │        13,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (1, 20, 50)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_19 (LSTM)                  │ (1, 50)                │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (1, 50)                │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (1, 8)                 │           408 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,808 (132.06 KB)

 Trainable params: 33,808 (132.06 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
# history_e1d1 = model_e1d1.fit(
#     X_train,
#     y_train,
#     epochs=100,
#     validation_data=(X_test, y_test),
#     batch_size=32,
#     verbose=1,
#     callbacks=[reduce_lr, early_stopping],
# )

In [42]:
class InputMonitor(keras.callbacks.Callback):
    def on_train_batch_begin(self, batch, logs=None):
        inputs, targets = self.model.inputs
        print(f"Starting batch {batch}, input shape: {inputs[0].shape}, target shape: {targets.shape}")
        
history_reg = regressor.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test, y_test),
    batch_size=32,
    verbose=1,
    callbacks=[reduce_lr, early_stopping, InputMonitor()],
)

Epoch 1/10


ValueError: not enough values to unpack (expected 2, got 1)

In [ ]:
# plot the test result
plt.plot(history_e1d1.history['loss'], label='train')
plt.plot(history_e1d1.history['val_loss'], label='test')
plt.legend()
plt.show()

In [ ]:
# plot the test result
plt.plot(history_reg.history["loss"], label="train")
plt.plot(history_reg.history["val_loss"], label="test")
plt.legend()
plt.show()

In [ ]:
pred = regressor.predict(X_test)
# plot the prediction of the test data
print(pred.shape)
# only grab the first features, turns from (4,1,8) to (4,1)
# pred = pred[:, :, 0]
# test = y_test[:, :, 0]

In [ ]:
print(y_test.shape)

# turn (1685, 1, 8) into (1685, 8)
test = y_test

In [ ]:
print(pred.shape)
print(test.shape)

# plot the prediction of the test data
import seaborn as sns
x_values = np.arange(pred.shape[0])
fig, ax = plt.subplots(
    figsize=(15, 5),
)

plt.title('LSTM autoregresser')

sns.scatterplot(x=x_values, y=test[:, 0], ax=ax, label='True')
sns.scatterplot(x=x_values, y=pred[:, 0], ax=ax, label='Predicted')
plt.show()

In [ ]:
pred = model_e1d1.predict(X_train)
# plot the prediction of the test data
print(pred)
# only grab the first features, turns from (4,1,8) to (4,1)
pred = pred[:, :, 0]
test = y_train[:, :, 0]

In [ ]:
print(pred.shape)
print(test.shape)

# plot the prediction of the test data
import seaborn as sns

x_values = np.arange(pred.shape[0])
fig, ax = plt.subplots(
    figsize=(15, 5),
)

plt.title("LSTM autoregresser")

sns.scatterplot(x=x_values, y=test[:, 0], ax=ax, label="True")
sns.scatterplot(x=x_values, y=pred[:, 0], ax=ax, label="Predicted")
plt.show()